Document Summarization using Iterative Refinement.

Use case -
1. Summarize large files 
2. Summarize Research Papers

In [ ]:
# !pip3 install sentence-transformers

In [ ]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, List


Load Document for Summarization 

In [80]:
def load_doc(file_path : str):
    document_loader = PyPDFLoader(file_path)
    documents = document_loader.load()
    # text_splitter = SentenceTransformersTokenTextSplitter(tokens_per_chunk=200, chunk_overlap=50)
    # splitted_doc = text_splitter.split_documents(documents=documents)
    return documents

In [ ]:
documents = load_doc("STM_8T.pdf") 
for doc in documents:
    print(doc.page_content)

April 2012 Doc ID 022351 Rev 2 1/22
PM0212
Programming manual
How to program the STM8TL5xxx
Flash program memory and data EEPROM
Introduction
This manual describes how to program Flash program memory and data EEPROM on 
STM8TL5xxx microcontrollers. It applies to STM8TL5xxx devices. It is intended to provide 
information to the programming tool manufacturers and to the customers who want to 
implement programming by themselves on their production line.
The in-circuit programming (ICP) method is used to update the content of Flash program 
memory and data EEPROM while the user software is not running. It uses the Single wire 
interface module (SWIM) to communicate between the programming tool and the device.
In contrast to the ICP method, in-application programming (IAP) can use any 
communication interface supported by the microcontroller (I/Os, SPI, USART, I
2C, USB, 
CAN...). IAP has been implemented for users who want their application software to update 
itself by re-programming the

The LLM

In [ ]:
api_key = <your api_key>
llm = ChatMistralAI(api_key=api_key, model_name= "mistral-large-latest")

The Nodes : generate, refine, END

In [64]:
async def generate_initial_summary(state : State):
    prompt = ChatPromptTemplate.from_template(""" Write a concise summary for the given content.
                                            {content}""")
    initial_summary_chain = prompt | llm
    initial_summary = await initial_summary_chain.ainvoke(input={"content":state["content"][0]})
    return {"summary": initial_summary, "index":1}


In [65]:
async def generate_refined_summary(state : State):
    refine_prompt = ChatPromptTemplate.from_template(""" With the given content and summary, refine the summary.
                                                    {content}

                                                    {summary}
                                                    """)
    refinement_chain = refine_prompt | llm 
    refined_summary = await refinement_chain.ainvoke(input={"summary" : state["summary"], "content" : state["content"][state["index"]]})

    return {"summary" : refined_summary, "index": state["index"]+1}


In [ ]:
class State(TypedDict):
    content : List[str]
    summary : str
    index : int

The Router

In [ ]:
def route(state:State):
    if state["index"] >= len(state["content"]):
        return END 
    else :
       return "generate_refined_summary"

In [68]:
graph_builder = StateGraph(State)
graph_builder.add_node("generate_initial_summary",generate_initial_summary)
graph_builder.add_node("generate_refined_summary", generate_refined_summary)
graph_builder.add_edge(START, "generate_initial_summary")
graph_builder.add_conditional_edges("generate_initial_summary", route)
graph_builder.add_conditional_edges("generate_refined_summary", route)

In [69]:
graph = graph_builder.compile()
graph.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnablePassthrough(), metadata=None), 'generate_initial_summary': Node(id='generate_initial_summary', name='generate_initial_summary', data=generate_initial_summary(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 'generate_refined_summary': Node(id='generate_refined_summary', name='generate_refined_summary', data=generate_refined_summary(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='generate_initial_summary', data=None, conditional=False), Edge(source='generate_initial_summary', target='__end__', data=None, conditional=False)])

INFERENCE

In [ ]:
async for step in graph.astream(
    {"content": [doc.page_content for doc in documents]},
    stream_mode="values",
):
    if summary := step.get("summary"): ## if the new summary is same as previous summary
        print(summary)

content='The document "PM0212: Programming Manual for STM8TL5xxx" provides guidelines for programming the Flash program memory and data EEPROM of STM8TL5xxx microcontrollers. It covers two programming methods: In-Circuit Programming (ICP) using the Single Wire Interface Module (SWIM), and In-Application Programming (IAP) using various communication interfaces. The manual is aimed at programming tool manufacturers and customers implementing programming on production lines. Key points include the use of ICP for updating memory when user software is not running, and IAP for self-updating applications while the software is running. Additional details on memory features and related documents are referenced for further information.' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 472, 'total_tokens': 627, 'completion_tokens': 155}, 'model_name': 'mistral-large-latest', 'model': 'mistral-large-latest', 'finish_reason': 'stop'} id='run--59a23730-be81-41fd-85c2-309bd527

In [73]:
summary.pretty_print()

================================== Ai Message ==================================

### Refined Summary

The "PM0212: Programming STM8 Flash Microcontrollers" document provides detailed instructions for programming the Flash program memory and data EEPROM of STM8 microcontrollers. It covers two primary programming methods:

1. **In-Circuit Programming (ICP)** using the Single Wire Interface Module (SWIM): This method allows for direct memory updates when the user software is inactive. The SWIM protocol, managed by hardware in the STM8 microcontrollers, offers non-intrusive debug capabilities, including program downloads, memory read/write operations, and jumping to specific memory addresses.

2. **In-Application Programming (IAP)** utilizing various communication interfaces: This method is used for self-updating applications while the software is running.

The document is aimed at tool manufacturers and production line implementers. It includes information on memory features, protection 